<div style="border-left:6px solid #14b8a6;padding:18px 22px;background:#f7f7f7">
<p style="margin:0;color:#14b8a6;letter-spacing:2px;font-size:12px">DATA MINING E INTELIGENCIA DE NEGOCIOS · ENCUENTRO VIRTUAL 1 · SÁBADO 31 DE OCTUBRE</p>
<h1 style="margin:6px 0 4px 0;font-weight:300">Del dato a la historia: el proyecto completo en una tarde, con datos reales B2B</h1>
<p style="margin:0;font-size:16px;color:#444">Data storytelling y estructura de un tablero</p>
<p style="margin:10px 0 0 0;color:#777;font-size:13px">Profe Alejandra Otero · Universidad EAN · 2026</p>
</div>

<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/data-mining-36993/notebooks/Encuentro1_B2B_Storytelling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Hoy no trabajamos con Mercado Central. Trabajamos con **datos reales**: las ventas a empresas (B2B) de un gran retailer colombiano entre abril y octubre de 2026, unas 20.000 líneas de factura. Los datos fueron **anonimizados** para poder usarlos en clase: no aparecen el nombre de la empresa, los NIT ni los nombres de los clientes, los códigos de sede ni la descripción de los productos.

En una tarde vamos a recorrer las cuatro fases del proyecto integrador, en versión exprés:

| Fase | Pregunta | Pasos |
|---|---|---|
| 1. Entender y limpiar (Guía 1) | ¿Qué datos tenemos y qué tan confiables son? | 0 a 2 |
| 2. Explorar (Guía 2) | ¿Qué está pasando con las ventas B2B? | 3 y 4 |
| 3. Segmentar (Guía 3) | ¿Quiénes son los clientes empresa? | 5 a 7 |
| 4. Contar la historia | ¿Qué debería decidir la gerencia? | 8 a 10 |

Al final cada persona construye un **tablero interactivo en HTML** con ayuda de IA.

### Datos del grupo

| Campo | Completar |
|---|---|
| Integrantes (nombre completo) | 1. <br>2. <br>3. <br>4. |
| Enlace de este cuaderno compartido como lector | (peguen aquí el enlace) |

## Antes de empezar

**1. Guarden su propia copia:** **Archivo > Guardar una copia en Drive**, y compártanla con su grupo como **Editor**.

**2. Descarguen la base desde Canvas.** Está en la actividad del encuentro: `base_b2b_anonimizada.xlsx`. **No la suban a ningún otro sitio ni la compartan fuera del curso**: aunque está anonimizada, son datos reales de una empresa.

**3. Gemini escribe el código; ustedes deciden e interpretan.** Usen el botón de **Gemini** de Colab y escriban sus propios prompts. Denle contexto: que trabajan en Python con pandas y cómo se llaman las tablas y columnas.

**4. Respeten los nombres** que pide cada paso (`df`, `ventas`, `clientes`...): el Paso 9 los necesita.

**5. Gobernanza de datos.** Antes de usar datos reales de una organización hay que: (a) tener autorización para usarlos, (b) quitar todo lo que identifique a personas o empresas (en Colombia lo exige la Ley 1581 de 2012 de protección de datos personales) y (c) proteger la información comercial sensible. Por eso esta base no trae nombres, NIT, sedes reales ni productos.

---
# Fase 1. Entender y limpiar

## Paso 0. Cargar la base (esta celda ya está lista: solo ejecútenla)

Al ejecutarla les aparece un botón **Elegir archivos**: seleccionen `base_b2b_anonimizada.xlsx`, que descargaron de Canvas. Deben ver **19.957 líneas y 14 columnas**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

subido = files.upload()
nombre = list(subido.keys())[0]
df = pd.read_excel(nombre, sheet_name="ventas_b2b", parse_dates=["fecha"])
diccionario = pd.read_excel(nombre, sheet_name="diccionario")

print(f"La base tiene {df.shape[0]:,} líneas de factura y {df.shape[1]} columnas")
display(diccionario)
df.head()

## Paso 1. Auditoría de datos

**Qué tienen que lograr:** revisar la calidad de la base antes de analizarla. Respondan con código:

1. ¿Qué tipo tiene cada columna? ¿Hay valores faltantes?
2. ¿Hay filas duplicadas si **ignoran** la columna `id_linea`? ¿Cuántas?
3. ¿Cuántas líneas tienen `valor_neto` igual a 0 y cuántas negativo? ¿Qué `tipo_documento` y `motivo` tienen las negativas?
4. ¿Qué hay en el grupo `BISER`? (Miren su `sublinea`.)
5. ¿Qué rango de fechas cubre la base? ¿Los meses están completos?

**Pistas técnicas:** `df.info()`, `df.isna().sum()`, `df.drop(columns="id_linea").duplicated().sum()`, filtros como `df[df["valor_neto"] < 0]`, `value_counts()`, `df["fecha"].min()`.

**Cómo saber que quedó bien:** encuentran 0 faltantes, varios cientos de líneas repetidas, 300 líneas en 0, 100 negativas (todas notas crédito) y 184 líneas de BISER.

In [ ]:
# PASO 1: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Completen la tabla de auditoría.

| Problema encontrado | ¿Cuántas líneas? | ¿Qué significa para el negocio? | Decisión (eliminar, separar, conservar) y por qué |
|---|---|---|---|
| Líneas repetidas | | *¿Error o el mismo producto comprado dos veces el mismo día?* | |
| Valor neto igual a 0 | | | |
| Notas crédito (valores negativos) | | | |
| Grupo BISER | | | |
| Abril incompleto | | | |

## Paso 2. Limpiar y separar

**Qué tienen que lograr:**

1. Crear la tabla `ventas` solo con las líneas que representan **ventas reales de producto**: facturas, con valor mayor a 0 y sin el grupo BISER.
2. Crear la tabla `devoluciones` con las notas crédito (las van a usar como indicador, no como ventas).
3. Mostrar cuántas líneas quedaron en cada tabla y cuántas se apartaron en total.

**Pistas técnicas:** filtros con `&`, `.copy()`.

**Cómo saber que quedó bien:** `ventas` queda con **19.373 líneas** y **1.500 clientes**; `devoluciones`, con 104 líneas.

In [ ]:
# PASO 2: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Escriban en dos líneas, como para un acta de calidad de datos, qué se apartó y por qué.

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

---
# Fase 2. Explorar

## Paso 3. Indicadores clave (KPI) y patrones

**Qué tienen que lograr:**

1. Los **KPI generales**: ventas netas totales, número de clientes, número de días con compra, ticket promedio por cliente y valor devuelto (notas crédito).
2. Las ventas por **unidad de negocio** y por **grupo de producto**, en valor y en número de líneas (porcentajes).
3. La evolución de las ventas por **mes**.
4. La **concentración**: ¿qué porcentaje de las ventas hace el 10 % de clientes que más compra?
5. Al menos **dos gráficas** con título que cuente la conclusión.

**Pistas técnicas:** `groupby()`, `sum()`, `nunique()`, `normalize=True`, `dt.to_period("M")`, `sort_values()`, `cumsum()`, `plot(kind="bar")` o `kind="barh"`.

**Cómo saber que quedó bien:** las ventas netas suman cerca de 2.744 millones de pesos. Fíjense en la diferencia entre el peso de cada grupo en **número de líneas** y en **valor**: ahí hay una historia.

In [ ]:
# PASO 3: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

## Paso 4. Hallazgos e insights

**Qué tienen que lograr:** convertir los números del Paso 3 en mensajes. Recuerden la cadena de valor de los datos: **dato** (qué pasó), **hallazgo** (qué patrón vemos), **insight** (por qué importa para el negocio) y **acción**.

**Respuesta del grupo.**

| # | Dato (con cifra) | Insight: por qué importa | Acción sugerida |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

---
# Fase 3. Segmentar

## Paso 5. De líneas de factura a clientes: la tabla RFM

Para segmentar necesitamos **una fila por cliente**. El modelo **RFM** resume a cada cliente en tres números:

| Variable | Qué mide | Cómo se calcula |
|---|---|---|
| **R**ecencia | Hace cuánto compró por última vez | Días entre su última compra y la última fecha de la base |
| **F**recuencia | Cuántas veces ha comprado | Número de días distintos con compra |
| **M**onto | Cuánto ha comprado | Suma de `valor_neto` |

**Qué tienen que lograr:**

1. Construir la tabla `clientes` (una fila por `id_cliente`) con `recencia`, `frecuencia` y `monto`.
2. Agregar al menos **dos variables más** que les parezcan útiles para describir a cada cliente (por ejemplo: porcentaje de su compra en electrónica, si compró en la Unidad B, número de sedes, si tuvo devoluciones).
3. Mostrar los descriptivos de la tabla.

**Pistas técnicas:** `ventas.groupby("id_cliente").agg(...)`, `(fecha_corte - fecha_max).dt.days`, `.isin()` para marcar devoluciones.

**Cómo saber que quedó bien:** `clientes` tiene **1.500 filas**. La mitad de los clientes compró **un solo día**.

In [ ]:
# PASO 5: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Cómo es el cliente típico (mediana) y qué tan distinto es del promedio? ¿Qué les dice eso?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 6. K-Means sobre RFM

**Qué tienen que lograr:**

1. Preparar las variables: el monto y la frecuencia tienen colas muy largas, así que usen su **logaritmo** (`np.log1p`) y luego **estandaricen** las tres (puntajes z).
2. Correr K-Means con **k de 2 a 6** y calcular la **inercia** y la **silueta** de cada k.
3. Elegir k con un criterio estadístico **y** uno de negocio (¿cuántos tipos de clientes puede atender diferente un equipo comercial?).
4. Entrenar el modelo final (`n_init=10`, `random_state=42`) y guardar el grupo en una columna `cluster` de la tabla `clientes`, numerada desde 1.

**Pistas técnicas:** `StandardScaler`, `KMeans`, `silhouette_score`, un ciclo `for k in range(2, 7)`.

**Cómo saber que quedó bien:** la silueta queda alrededor de 0,3 y baja poco a poco al subir k.

In [ ]:
# PASO 6: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Qué k eligieron y por qué (criterio estadístico y de negocio)?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 7. Perfiles y nombres de los segmentos

**Qué tienen que lograr:**

1. Una tabla por clúster con: número de clientes, % de clientes, **mediana** de recencia, frecuencia y monto, y **% de las ventas totales** que aporta.
2. Una gráfica que compare el **% de clientes** contra el **% de ventas** de cada segmento.
3. Un nombre y una acción para cada segmento.

**Pistas técnicas:** `groupby("cluster").agg(...)`, `median`, barras agrupadas.

**Cómo saber que quedó bien:** uno de los segmentos tiene pocos clientes pero cerca de la mitad de las ventas.

In [ ]:
# PASO 7: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.**

| Clúster | Nombre del segmento | % clientes | % ventas | Cómo es (R, F, M) | Acción comercial |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |

### Reto opcional (anticipo de la Guía 4): ¿se puede saber desde la primera compra si un cliente va a volver?

Si les sobra tiempo: tomen solo los clientes cuya primera compra fue **entre abril y junio** (así todos tuvieron al menos cuatro meses para volver), marquen como `recurrente = 1` a quienes compraron en dos o más días y entrenen un árbol de decisión pequeño con variables de su **primera compra** (valor, número de líneas, si incluyó mercado o electrónica). Comparen su exactitud con la línea base. Spoiler: no es fácil, y eso también es un hallazgo que vale la pena contar.

In [ ]:
# RETO OPCIONAL: árbol de decisión de recurrencia.

---
# Fase 4. Contar la historia

## Paso 8. La historia en tres mensajes

Un tablero no es una colección de gráficas: es una **respuesta**. Antes de construirlo, escriban la historia.

**Qué tienen que lograr:**

1. **El mensaje principal (BLUF, *bottom line up front*):** una sola frase con la conclusión y la decisión que recomiendan. Va arriba del tablero.
2. **Tres mensajes de apoyo**, cada uno con su cifra. Cada uno será una zona del tablero.
3. **Para quién** es el tablero (gerencia comercial B2B) y **qué decisión** debe tomar con él.

**Respuesta del grupo.**

- **Para quién y qué decisión:**
- **Mensaje principal (BLUF):**
- **Mensaje de apoyo 1 (Fase 1 o 2, con cifra):**
- **Mensaje de apoyo 2 (Fase 2, con cifra):**
- **Mensaje de apoyo 3 (Fase 3, con cifra):**

## Paso 9. Resumen para el tablero (esta celda ya está lista: solo ejecútenla)

Junta los números clave en un archivo pequeño, `resumen_b2b.json`, que es lo **único** que le van a pasar a la IA para construir el tablero. Así la base completa nunca sale de Colab: **no le suban la base a ninguna IA externa**.

Necesita que existan `df`, `ventas`, `devoluciones` y `clientes` (con la columna `cluster`).

In [ ]:
import json

def _r(x, n=4):
    return None if pd.isna(x) else round(float(x), n)

corte = ventas["fecha"].max()
total = ventas["valor_neto"].sum()
por_cliente = ventas.groupby("id_cliente")["valor_neto"].sum().sort_values(ascending=False)
top10 = por_cliente.head(max(1, int(len(por_cliente) * 0.10))).sum() / total

resumen = {
    "caso": "Ventas B2B de un gran retailer colombiano (datos reales anonimizados)",
    "periodo": {"desde": str(ventas["fecha"].min().date()), "hasta": str(corte.date())},
    "calidad_datos": {
        "lineas_originales": int(len(df)),
        "lineas_venta_analizadas": int(len(ventas)),
        "lineas_apartadas": int(len(df) - len(ventas)),
        "notas_credito": int(len(devoluciones)),
    },
    "kpi": {
        "ventas_netas": _r(total, 0),
        "clientes": int(ventas["id_cliente"].nunique()),
        "ticket_promedio_cliente": _r(total / ventas["id_cliente"].nunique(), 0),
        "valor_devuelto": _r(devoluciones["valor_neto"].sum(), 0),
        "participacion_top10_clientes": _r(top10),
    },
    "ventas_por_mes": {str(k): _r(v, 0) for k, v in ventas.groupby(ventas["fecha"].dt.to_period("M"))["valor_neto"].sum().items()},
    "ventas_por_unidad": {k: _r(v / total) for k, v in ventas.groupby("unidad_negocio")["valor_neto"].sum().items()},
    "grupos": [
        {"grupo": g, "pct_valor": _r(r["valor_neto"] / total), "pct_lineas": _r(r["lineas"] / len(ventas))}
        for g, r in ventas.groupby("grupo").agg(valor_neto=("valor_neto", "sum"), lineas=("valor_neto", "size"))
        .sort_values("valor_neto", ascending=False).iterrows()
    ],
}
seg = []
for c, g in clientes.groupby("cluster"):
    seg.append({
        "cluster": str(c), "clientes": int(len(g)), "pct_clientes": _r(len(g) / len(clientes)),
        "pct_ventas": _r(g["monto"].sum() / clientes["monto"].sum()),
        "mediana_recencia_dias": _r(g["recencia"].median(), 1),
        "mediana_frecuencia": _r(g["frecuencia"].median(), 1),
        "mediana_monto": _r(g["monto"].median(), 0),
    })
resumen["segmentos"] = seg

with open("resumen_b2b.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print(json.dumps(resumen, ensure_ascii=False, indent=2))
files.download("resumen_b2b.json")

## Paso 10. El tablero interactivo con IA

Cada persona construye su tablero en HTML con la IA que prefiera (Gemini, Claude, ChatGPT u otra). El prompt está en la página de la actividad en Canvas: cópienlo, **completen los espacios** con lo que escribieron en los Pasos 7 y 8 y **peguen al final el contenido de `resumen_b2b.json`**.

Cuando la IA les entregue el código:

1. Cópienlo en un archivo de texto y guárdenlo como `tablero_b2b.html` (en el Bloc de notas o TextEdit, en modo texto plano).
2. Ábranlo con doble clic en su navegador y revisen: ¿el título cuenta la conclusión?, ¿cada gráfica responde una pregunta?, ¿el color destaca solo lo importante?, ¿las cifras coinciden con su cuaderno?
3. Pídanle a la IA los ajustes que necesiten, uno a la vez.

**Respuesta del grupo.** ¿Qué tuvieron que corregirle a la IA en el tablero (cifras, diseño, mensajes)?

> Escriban aquí.

## Bitácora de uso de IA

| Paso | Prompt principal (cópienlo tal cual) | ¿Funcionó a la primera? Qué corrigieron |
|---|---|---|
| 1 | | |
| 5 | | |
| 6 | | |
| 10 | | |

## Qué se sube a Canvas (cada persona, en la actividad del encuentro)

1. **El tablero** `tablero_b2b.html`.
2. **El cuaderno** de Colab (.ipynb) del grupo, con el enlace compartido escrito al inicio.
3. **Su reflexión individual** (en el cuadro de texto de la entrega, 150 a 200 palabras): ¿qué decisión le recomendaría a la gerencia comercial B2B y por qué?, y ¿qué aprendieron hoy sobre trabajar con datos reales?

<p style="color:#777;font-size:13px">Entrega hasta hoy sábado 31 de octubre a las 11:59 p. m. Profe Alejandra Otero.</p>